### Import packages

In [25]:
import os
import time
import yaml
import numpy as np
import os
from typing import List, Dict

# from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer

# from langchain_google_genai import ChatGoogleGenerativeAI  # Commenting out due to version conflict
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pypdf import PdfReader

import os
import re
import psycopg2
from datetime import datetime
from google.api_core.client_options import ClientOptions
from google.cloud import documentai_v1 as documentai
from google.cloud import aiplatform
import vertexai
from vertexai.language_models import TextEmbeddingModel


### Import the keys

In [ ]:

import os
from dotenv import load_dotenv
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY environment variable not set. Please check your .env file.")
# pinecone_api_key = os.environ.get('PINECONE_API_KEY')

project_id = os.environ.get("GOOGLE_CLOUD_PROJECT") or "intelligent-cc-optimization"
location = "us"
processor_id = os.environ.get("DOCUMENT_AI_PROCESSOR_ID") or "3ccf5d1678ae6859"
bucket_name = "my_storage_files"
file_name = "CC_project/CC_documents/AE_platinum_travel.pdf"
db_host = os.environ.get("DB_HOST") or "34.10.133.69" 
db_name = os.environ.get("DB_NAME") or "postgres"
db_user = os.environ.get("DB_USER") or "postgres"
db_pass = os.environ.get("DB_PASS") or "rag123"

### Read the PDF Document and Extract Text

In [31]:


def read_pdf_text_only(pdf_path: str) -> str:
    """
    Reads a PDF file and returns only its text content, 
    completely ignoring embedded images.
    """
    try:
        reader = PdfReader(pdf_path)
        full_text = []
        
        for i, page in enumerate(reader.pages):
            # extract_text() naturally skips images and extracts the text layer
            text = page.extract_text()
            if text:
                full_text.append(text)
                
        return "\n\n".join(full_text)
        
    except Exception as e:
        return f"Error reading PDF: {e}"

# Example Usage: gcs_uri=f"gs://{bucket_name}/{file_name}"
document_text = read_pdf_text_only(".\\data\\raw_pdfs\\HDFC_diners_black.pdf")
# document_text = read_pdf_text_only(gcs_uri=f"gs://{bucket_name}/{file_name}")
print(document_text)

 3rd July 2024 
Terms & Conditions – Diners Club Black 
 Welcome Benefits: 
Diners Black: Club Marriott, Swiggy One (3 months), MakeMyTrip Black Gold Tier, Amazon Prime & Times 
Prime Subscription 
 
Avail Complimentary annual memberships from the above partners by meeting either of two below  
conditions- 
 
Product Condition 1 Condition 2 
Diners Black Spending Rs. 1.5 lakhs within the first 90 days (or) Joining fee realization 
 
The benefits will be communicated on your HDFC bank registered Mobile Number via SMS  / Email ID 
within 60 days of condition being met. From the date of receipt of communication from the bank, the  
customer can download the benefits within 60 days, post which the benefits expire. 
 
Welcome benefits can only be availed as on New issued cards / Upgraded cards. Welcome benefit is not 
applicable on Renewal Card / Reissued Cards. 
 
How to track/avail 
• Visit Diners Club Black Smartbuy Milestone Page: https://offers.smartbuy.hdfcbank.com/diners >> 
Diners C

### Create chunks of extracted text

In [19]:

CHUNK_SIZE_WORDS = 500
CHUNK_OVERLAP_WORDS = 75

def split_words_with_overlap(text: str, size: int, overlap: int) -> List[str]:
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + size
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        start = end - overlap

    return chunks

chunk_text = split_words_with_overlap(document_text, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)

In [22]:
chunk_text[1]

'Welcome benefits will also be reversed / adjusted. • Transactions are tracked as per the posted/settled date from third party/merchant to the bank and the posting/settlement date will be considered as contribution to the spend accrual. • All Terms & Conditions as prescribed for the membership by the respective merchant is applicable. • Customer will have to visit the respective merchant website to activate the membership using the coupon code. Annual Benefits: Above Complimentary memberships can also be availed at the time of card anniversary (Card anniversary is 12 months from card open month) subject to achieving annual spend milestone of Rs 8 Lakhs for Diners Club Black. The merchants as applicable in the offer, at the time of card anniversary can be availed. • Customers eligible for Annual membership benefits will be communicated on HDFC Bank registered Mobile Number via SMS and Email ID within 60 days from the date of card anniversary. • From the date of receipt of communication 

In [26]:


def generate_vertex_embeddings(project_id, location, text_list):
    """Generates 768-dimensional dense vector embeddings using text-embedding-004."""
    # If the location passed is 'us', override it to 'us-central1' for Vertex AI compatibility
    vertex_location = "us-central1" if location == "us" else location
    vertex_location = "europe-west1" if location == "eu" else vertex_location
    
    # Initialize using the valid regional endpoint string
    vertexai.init(project=project_id, location=vertex_location)
    
    model = TextEmbeddingModel.from_pretrained("text-embedding-004")
    
    print(f"Generating vectors for {len(text_list)} text blocks via Vertex AI ({vertex_location})...")
    embeddings = model.get_embeddings(text_list)
    
    return [emb.values for emb in embeddings]

In [27]:
computed_vectors = generate_vertex_embeddings(project_id, location, chunk_text)

Generating vectors for 17 text blocks via Vertex AI (us-central1)...


In [29]:
len(computed_vectors)

17

In [ ]:
import json
from uuid import uuid4

def store_in_postgres(db_config, metadata, chunks, vectors):
    """Safely stores chunks with enhanced metadata including document_id, ChunkID, and JSON metadata."""
    conn = psycopg2.connect(
        host=db_config['host'],
        database=db_config['dbname'],
        user=db_config['user'],
        password=db_config['password'],
        port=db_config['port']
    )
    cursor = conn.cursor()
    
    insert_query = """
    INSERT INTO cc_reward_chunks (
        document_id, chunk_id, card_name, issuer_bank, chunk_text, 
        page_number, effective_date, source_url, embedding, chunk_metadata
    ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """
    
    print(f"Inserting entries cleanly into PostgreSQL table...")
    try:
        for idx, chunk in enumerate(chunks):
            # Generate unique ChunkID
            chunk_id = str(uuid4())
            
            # Create JSON metadata for the chunk
            chunk_metadata = {
                "chunk_index": idx,
                "chunk_size_words": len(chunk['text'].split()),
                "created_at": datetime.utcnow().isoformat(),
                "document_type": metadata.get('document_type', 'Unknown'),
                "source_metadata": {
                    "issuer": metadata.get('issuer'),
                    "card_name": metadata.get('card_name'),
                    "url": metadata.get('source_url')
                }
            }
            
            cursor.execute(insert_query, (
                metadata['document_id'],
                chunk_id,
                metadata['card_name'],
                metadata['issuer'],
                chunk['text'],
                chunk['page_number'],
                metadata['effective_date'],
                metadata['source_url'],
                vectors[idx],  # pgvector handles list to array natively
                json.dumps(chunk_metadata)  # Store metadata as JSON string
            ))
        conn.commit()
        print(f"🚀 Successfully saved {len(chunks)} chunks to database!")
    except Exception as e:
        conn.rollback()
        print(f"Error executing database transactions: {e}")
    finally:
        cursor.close()
        conn.close()

In [ ]:
GCP_PROJECT = "intelligent-cc-optimization"
GCP_LOCATION = "us" # Multi-region or regional endpoint matching your processor
PROCESSOR_ID = "your_processor_hex_string_here"

BUCKET_NAME = "my_storage_files"
FILE_NAME = "CC_project/CC_documents/AE_platinum_travel.pdf"

# 2. Extract Document Metadata Context attributes cleanly from naming conventions
# e.g., 'AE_platinum_travel.pdf' -> issuer: 'AE', card_name: 'platinum_travel'
base_file = os.path.basename(FILE_NAME)
name_split = base_file.replace(".pdf", "").split("_", 1)

inferred_issuer = name_split[0].upper() if len(name_split) > 0 else "UNKNOWN"
inferred_card = name_split[1].replace("_", " ").title() if len(name_split) > 1 else "Generic Card"

doc_metadata = {
    "document_id": f"DOC_{int(datetime.utcnow().timestamp())}", # Unique anchor ID string
    "card_name": inferred_card,
    "issuer": inferred_issuer,  # issuer_bank in database
    "document_type": "Terms and Conditions",
    "effective_date": "2026-01-01",  # Set static or parse via regex patterns
    "source_url": f"https://storage.googleapis.com/{bucket_name}/{file_name}".replace("//", "/")
}

# 3. Cloud SQL Connection Configurations
db_credentials = {
    "host": db_host, # Your public IP or 127.0.0.1 if running through Cloud SQL Proxy
    "dbname": db_name,
    "user": db_user,
    "password": db_pass,
    "port": "5432"
}

# --- PIPELINE WORKFLOW EXECUTION ---
# Step A: Parse PDF and create text chunks
extracted_text = read_pdf_text_only(".\\data\\raw_pdfs\\HDFC_diners_black.pdf")
print(f"Extracted {len(extracted_text)} characters from PDF")

if extracted_text:
    # Step B: Split text into chunks
    text_chunks = split_words_with_overlap(extracted_text, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)
    print(f"Created {len(text_chunks)} chunks")
    
    # Step C: Structure chunks as dictionaries with metadata
    structured_chunks = [
        {
            "text": chunk, 
            "page_number": 1  # Improve page number tracking if available from PDF
        }
        for chunk in text_chunks
    ]
    
    # Step D: Generate embeddings
    computed_vectors = generate_vertex_embeddings(project_id, location, text_chunks)
    print(f"Generated {len(computed_vectors)} embeddings of dimension {len(computed_vectors[0])}")
    
    # Step E: Save to PostgreSQL Database with enhanced metadata
    store_in_postgres(db_credentials, doc_metadata, structured_chunks, computed_vectors)
else:
    print("Pipeline aborted: Document text parsing step returned empty sequences.")

Extracted 41552 characters from PDF
Created 17 chunks
Generating vectors for 17 text blocks via Vertex AI (us-central1)...
Generated 17 embeddings
Inserting entries cleanly into PostgreSQL table...
🚀 Successfully saved database assets!


In [50]:
len(computed_vectors)
len(computed_vectors[1])

768